In [ ]:
# ============================================================================
# INSTALLATION AND SETUP
# ============================================================================

# Install required packages
!pip install dspy-ai huggingface_hub networkx sentence-transformers pandas --quiet

print("✓ Packages installed successfully!")

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import json
import networkx as nx
from typing import List, Dict, Set, Tuple
import dspy
from sentence_transformers import SentenceTransformer
import numpy as np
from collections import defaultdict
import pandas as pd
import os
from huggingface_hub import InferenceClient

print("✓ All imports successful!")

In [ ]:
import os
import dspy
 
lm = dspy.LM(
    model="openai/gpt-4o",  # OpenRouter model path
    api_key="API_KEY", #put the api key
    api_base="https://openrouter.ai/api/v1",
    max_tokens=10000,
    temperature=0.3
)
dspy.settings.configure(lm=lm)
print("✓ Model configured via OpenRouter!")

In [ ]:
# ============================================================================
# DATA LOADING FUNCTION
# ============================================================================

def load_json_data(filepath: str) -> List[Dict]:
    """
    Load JSON data from file
    
    Expected format:
    [
        {
            "Article Title": [],
            "Article Text": "text here...",
            "Concept": ["concept1", "concept2"]
        },
        ...
    ]
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"✓ Loaded {len(data)} documents from {filepath}")
    return data


# Test with sample data
sample_data = [
    {
        "Article Title": [],
        "Article Text": "excellent phone, excellent service.",
        "Concept": []
    },
    {
        "Article Title": [],
        "Article Text": "i am a business user who heavily depend on mobile service.",
        "Concept": ["service"]
    }
]

print("✓ Sample data ready for testing")
print(f"  Sample has {len(sample_data)} documents")

In [ ]:
# ============================================================================
# LOAD ONE SPECIFIC DATASET
# ============================================================================


filepath = 'DATASET_PATH' #put dataset path

# Load the data
data = load_json_data(filepath)

print(f"✓ Loaded {len(data)} documents")
print(f"\nFirst document preview:")
print(f"  Keys: {list(data[0].keys())}")
print(f"  Text: {data[0].get('Article Text', '')}...")
print(f"  Concepts: {data[0].get('Concept', [])}")

In [ ]:
# ============================================================================
# PROCESS ALL THE ARTICLES
# ============================================================================

print("=" * 80)
print("PROCESSING ALL THE ARTICLES")
print("=" * 80)

# data is now correctly loaded as a list of dicts
individual_articles = []

for idx, item in enumerate(data):
    article = {
        'id': idx,
        'Article Text': item['Article Text'],
        #'Concept': item['Concept'] if item['Concept'] else []
    }
    individual_articles.append(article)

print(f"\n✓ Created list of {len(individual_articles)} individual articles")

# Show first 3
print(f"\nFirst 3 articles:")
print("-" * 80)
for i in range(min(3, len(individual_articles))):
    article = individual_articles[i]
    print(f"\nArticle {article['id']}:")
    print(f"  Text: {article['Article Text'][:80]}...")
    #print(f"  Concepts: {article['Concept']}")

In [ ]:
# ============================================================================
# DEFINE SIGNATURES
# ============================================================================

class EntityExtractor(dspy.Signature):
    """Extract key entities from the given text. Extracted entities are nouns, 
    verbs, or adjectives, particularly regarding sentiment. This is for an 
    extraction task, please be thorough and accurate to the reference text.
    
    Return ONLY a valid JSON list format: ["entity1", "entity2", "entity3"]
    """
    
    text = dspy.InputField(desc="The text to extract entities from")
    entities = dspy.OutputField(desc="List of extracted entities in JSON format")

class RelationExtractor(dspy.Signature):
    """Extract subject-predicate-object triples from the assistant message. 
    A predicate (1-3 words) defines the relationship between the subject and 
    object. Relationship may be fact or sentiment based on assistant's message. 
    Subject and object are entities. Entities provided are from the assistant 
    message and prior conversation history, though you may not need all of them. 
    This is for an extraction task, please be thorough, accurate, and faithful 
    to the reference text.
    
    Return ONLY valid JSON format: [["subject1", "predicate1", "object1"], ["subject2", "predicate2", "object2"]]
    """
    
    text = dspy.InputField(desc="The text to extract relations from")
    entities = dspy.InputField(desc="List of available entities")
    triples = dspy.OutputField(desc="List of [subject, predicate, object] triples in JSON format")

print("✓ Signatures defined")

In [ ]:
# ============================================================================
# CREATE ENTITY EXTRACTOR
# ============================================================================

class ExtractEntities(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(EntityExtractor)
    
    def forward(self, text: str) -> List[str]:
        if not text or len(text.strip()) < 3:
            return []
            
        result = self.extract(text=text)
        
        try:
            entities_text = result.entities.strip()
            
            if '[' in entities_text and ']' in entities_text:
                start = entities_text.find('[')
                end = entities_text.rfind(']') + 1
                entities_text = entities_text[start:end]
            
            entities = json.loads(entities_text)
            
            if isinstance(entities, list):
                return [str(e).lower().strip() for e in entities if e and len(str(e).strip()) > 1]
            return []
            
        except:
            try:
                entities_text = result.entities.strip()
                if entities_text.startswith('['):
                    entities_text = entities_text[1:]
                if entities_text.endswith(']'):
                    entities_text = entities_text[:-1]
                
                entities = []
                for item in entities_text.split(','):
                    item = item.strip(' "\'\n\t')
                    if item and len(item) > 1:
                        entities.append(item.lower())
                
                return entities[:50]
            except:
                return []

entity_extractor = ExtractEntities()
print("✓ Entity Extractor created")

In [ ]:
# ============================================================================
# CREATE RELATION EXTRACTOR
# ============================================================================

class ExtractRelations(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(RelationExtractor)
    
    def forward(self, text: str, entities: List[str]) -> List[Tuple[str, str, str]]:
        if not entities or not text:
            return []
        
        entities_subset = entities[:30]
        entities_str = json.dumps(entities_subset)
        
        result = self.extract(text=text, entities=entities_str)
        
        try:
            triples_text = result.triples.strip()
            
            if '[' in triples_text and ']' in triples_text:
                start = triples_text.find('[')
                end = triples_text.rfind(']') + 1
                triples_text = triples_text[start:end]
            
            triples = json.loads(triples_text)
            
            normalized_triples = []
            for triple in triples:
                if isinstance(triple, (list, tuple)) and len(triple) == 3:
                    s, p, o = triple
                    s = str(s).lower().strip()
                    p = str(p).lower().strip()
                    o = str(o).lower().strip()
                    
                    if s and p and o and s != o:
                        normalized_triples.append((s, p, o))
            
            return normalized_triples
            
        except Exception as e:
            return []

relation_extractor = ExtractRelations()
print("✓ Relation Extractor created")

In [ ]:
# ============================================================================
# CLUSTERING SIGNATURES (DEFINE FIRST!)
# ============================================================================

class ClusterValidator(dspy.Signature):
    """Verify if these entities belong in the same cluster.
    A cluster should contain entities that are the same in meaning, with different:
    - tenses, plural forms, stem forms, upper/lower cases
    Or entities with close semantic meanings.
    
    Return ONLY valid JSON format: ["entity1", "entity2", "entity3"]
    Return only entities you are confident belong together.
    If not confident, return empty list [].
    """
    
    entities = dspy.InputField(desc="Entities to validate")
    valid_cluster = dspy.OutputField(desc="Validated cluster in JSON format")

print("✓ ClusterValidator Signature defined")

In [ ]:
# ============================================================================
# SEMANTIC SIMILARITY CLUSTERING
# ============================================================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class SemanticEntityClustering(dspy.Module):
    def __init__(self, similarity_threshold=0.75):
        super().__init__()
        self.validator = dspy.ChainOfThought(ClusterValidator)
        
        # Load embedding model
        print("Loading sentence transformer model...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✓ Model loaded")
        
        self.similarity_threshold = similarity_threshold
    
    def _parse_cluster(self, text: str) -> List[str]:
        """Parse cluster from LLM response"""
        try:
            text = text.strip()
            if '[' in text and ']' in text:
                start = text.find('[')
                end = text.rfind(']') + 1
                text = text[start:end]
            
            cluster = json.loads(text)
            if isinstance(cluster, list):
                return [str(e).lower().strip() for e in cluster if e]
            return []
        except:
            return []
    
    def _get_semantic_clusters(self, entities: List[str]) -> List[List[str]]:
        """Group entities by semantic similarity using embeddings"""
        
        if len(entities) == 0:
            return []
        
        # Get embeddings for all entities
        embeddings = self.model.encode(entities)
        
        # Compute pairwise cosine similarity
        similarity_matrix = cosine_similarity(embeddings)
        
        # Find clusters using similarity threshold
        clusters = []
        remaining = set(range(len(entities)))
        
        for i in range(len(entities)):
            if i not in remaining:
                continue
            
            # Find all entities similar to this one
            cluster_indices = [i]
            remaining.discard(i)
            
            for j in range(i + 1, len(entities)):
                if j not in remaining:
                    continue
                
                # Check if similar enough
                if similarity_matrix[i][j] >= self.similarity_threshold:
                    cluster_indices.append(j)
                    remaining.discard(j)
            
            # Convert indices to entity names
            cluster = [entities[idx] for idx in cluster_indices]
            
            # Only keep clusters with 2-4 entities
            if 2 <= len(cluster) <= 4:
                clusters.append(cluster)
            elif len(cluster) == 1:
                # Keep singletons for later
                pass
        
        return clusters
    
    def forward(self, entities: List[str]) -> Dict[str, List[str]]:
        """Semantic clustering with LLM validation"""
        
        print(f"Starting semantic clustering with {len(entities)} entities...")
        print(f"  Similarity threshold: {self.similarity_threshold}")
        
        # Remove duplicates
        unique_entities = list(set(entities))
        
        # Step 1: Find semantic clusters using embeddings
        print("  Computing semantic similarities...")
        potential_clusters = self._get_semantic_clusters(unique_entities)
        
        print(f"  Found {len(potential_clusters)} potential clusters")
        
        # Step 2: Validate with LLM
        validated_clusters = {}
        cluster_id = 0
        clustered_entities = set()
        
        for cluster in potential_clusters:
            try:
                # Ask LLM to validate
                validation = self.validator(entities=json.dumps(cluster))
                validated = self._parse_cluster(validation.valid_cluster)
                
                if validated and len(validated) >= 2:
                    cluster_label = validated[0]
                    validated_clusters[cluster_label] = validated
                    
                    for entity in validated:
                        clustered_entities.add(entity)
                    
                    print(f"  ✓ Cluster {cluster_id}: {validated}")
                    cluster_id += 1
                else:
                    # LLM rejected - add as singletons
                    for entity in cluster:
                        if entity not in clustered_entities:
                            validated_clusters[entity] = [entity]
                            clustered_entities.add(entity)
            except:
                # Error - add as singletons
                for entity in cluster:
                    if entity not in clustered_entities:
                        validated_clusters[entity] = [entity]
                        clustered_entities.add(entity)
        
        # Step 3: Add all remaining entities as singletons
        for entity in unique_entities:
            if entity not in clustered_entities:
                validated_clusters[entity] = [entity]
        
        multi = sum(1 for v in validated_clusters.values() if len(v) > 1)
        print(f"✓ Semantic clustering complete: {len(validated_clusters)} total clusters")
        print(f"  Multi-entity clusters: {multi}")
        print(f"  Singleton entities: {len(validated_clusters) - multi}")
        
        return validated_clusters

# Create semantic clusterer with different thresholds
entity_clusterer_semantic = SemanticEntityClustering(similarity_threshold=0.75)
print("\n✓ Semantic Entity Clustering Module created")

In [ ]:
# ============================================================================
# FINAL KGGEN WITH SEMANTIC CLUSTERING
# ============================================================================

class KGGenSemantic:
    def __init__(self, similarity_threshold=0.75):
        self.entity_extractor = entity_extractor
        self.relation_extractor = relation_extractor
        self.entity_clusterer = SemanticEntityClustering(similarity_threshold=similarity_threshold)
        self.graph = nx.DiGraph()
        self.entity_clusters = {}
        
    def generate_from_json(self, json_data: List[Dict], max_docs: int = None) -> nx.DiGraph:
        """Generate KG from JSON dataset"""
        all_entities = set()
        all_relations = []
        
        if max_docs:
            json_data = json_data[:max_docs]
        
        print(f"Processing {len(json_data)} documents...")
        print("=" * 80)
        
        for idx, item in enumerate(json_data):
            text = item.get('Article Text', '')
            concepts = item.get('Concept', [])
            
            if not text or len(text.strip()) < 5:
                continue
            
            try:
                # Extract entities
                entities = self.entity_extractor(text)
                all_entities.update(entities)
                
                # Add concepts
                #for concept in concepts:
                    #if concept and isinstance(concept, str):
                        #all_entities.add(concept.lower().strip())
                
                # Extract relations
                relations = self.relation_extractor(text, list(all_entities))
                all_relations.extend(relations)
                
                if (idx + 1) % 20 == 0:
                    print(f"  {idx + 1}/{len(json_data)} docs | {len(all_entities)} entities | {len(all_relations)} relations")
                    
            except Exception as e:
                continue
        
        print(f"\n✓ Extraction complete!")
        print(f"  Total entities: {len(all_entities)}")
        print(f"  Total relations: {len(all_relations)}")
        
        # Build graph
        for subj, pred, obj in all_relations:
            self.graph.add_edge(subj, obj, relation=pred)
        
        print(f"  Graph nodes: {len(self.graph.nodes())}")
        print(f"  Graph edges: {len(self.graph.edges())}")
        
        return self.graph
    
    def cluster_entities(self):
        """Semantic clustering with embeddings"""
        nodes = list(self.graph.nodes())
        
        if len(nodes) == 0:
            print("No nodes to cluster!")
            return self.graph
        
        print(f"\n{'='*80}")
        print(f"SEMANTIC CLUSTERING: {len(nodes)} ENTITIES")
        print(f"{'='*80}")
        
        self.entity_clusters = self.entity_clusterer(nodes)
        
        # Map entities
        entity_mapping = {}
        for cluster_label, cluster_entities in self.entity_clusters.items():
            for entity in cluster_entities:
                entity_mapping[entity] = cluster_label
        
        # Rebuild graph
        new_graph = nx.DiGraph()
        for u, v, data in self.graph.edges(data=True):
            new_u = entity_mapping.get(u, u)
            new_v = entity_mapping.get(v, v)
            relation = data.get('relation', 'related_to')
            
            if new_u == new_v:
                continue
            
            if not new_graph.has_edge(new_u, new_v):
                new_graph.add_edge(new_u, new_v, relation=relation)
        
        self.graph = new_graph
        
        print(f"\n✓ Clustering complete!")
        print(f"  Final nodes: {len(self.graph.nodes())}")
        print(f"  Final edges: {len(self.graph.edges())}")
        
        return self.graph
    
    def save_graph(self, filepath: str):
        data = nx.node_link_data(self.graph)
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"✓ Saved to {filepath}")
    
    def export_triples(self, filepath: str):
        triples = []
        for u, v, data in self.graph.edges(data=True):
            triples.append({
                'subject': u,
                'predicate': data.get('relation', 'related_to'),
                'object': v
            })
        import pandas as pd
        df = pd.DataFrame(triples)
        df.to_csv(filepath, index=False)
        print(f"✓ Exported to {filepath}")

# Initialize semantic KGGen (threshold 0.75 = balanced)
kg_gen_semantic = KGGenSemantic(similarity_threshold=0.75)
print("\n✓ KGGen with Semantic Clustering (Embeddings + LLM) created!")

In [ ]:
# ============================================================================
# GENERATE KG FOR FIRST 50 ARTICLES
# ============================================================================

print("\n" + "=" * 80)
print("GENERATING KNOWLEDGE GRAPHS FOR ALL THE ARTICLES")
print("=" * 80)

all_article_kgs = []

# Process only first 50 articles (or fewer if there aren't 50)
first_50_articles = individual_articles
total = len(first_50_articles)
print(f"\nProcessing {total} articles...\n")

for idx, article in enumerate(first_50_articles):
    article_id = article['id']
    text = article['Article Text']
    #concepts = article['Concept']
    
    # Skip empty articles
    if not text or len(text.strip()) < 5:
        all_article_kgs.append({
            'id': article_id,
            'text': text,
            #'concepts': concepts,
            'graph': nx.DiGraph(),
            'entities': [],
            'relations': [],
            'num_nodes': 0,
            'num_edges': 0
        })
        continue
    
    try:
        # Extract entities for THIS article
        entities = entity_extractor(text)
        entities = list(set(entities))  # Remove duplicates
        
        # Extract relations for THIS article
        if entities:
            relations = relation_extractor(text, entities)
        else:
            relations = []
        
        # Build graph for THIS article
        graph = nx.DiGraph()
        for subj, pred, obj in relations:
            graph.add_edge(subj, obj, relation=pred)
        
        # Store everything for this article
        all_article_kgs.append({
            'id': article_id,
            'text': text,
            #'concepts': concepts,
            'graph': graph,
            'entities': entities,
            'relations': relations,
            'num_nodes': len(graph.nodes()),
            'num_edges': len(graph.edges())
        })
        
        # Progress update
        if (idx + 1) % 10 == 0:
            print(f"✓ Processed {idx + 1}/{total} articles...")
        
    except Exception as e:
        print(f"✗ Article {article_id}: Error - {str(e)}")
        all_article_kgs.append({
            'id': article_id,
            'text': text,
            #'concepts': concepts,
            'graph': nx.DiGraph(),
            'entities': [],
            'relations': [],
            'num_nodes': 0,
            'num_edges': 0
        })

print(f"\n" + "=" * 80)
print("✓ KNOWLEDGE GRAPH GENERATION COMPLETE!")
print("=" * 80)
print(f"Total articles processed: {len(all_article_kgs)}")
print(f"Articles with graphs: {sum(1 for kg in all_article_kgs if kg['num_edges'] > 0)}")
print(f"Articles without graphs: {sum(1 for kg in all_article_kgs if kg['num_edges'] == 0)}")

# Statistics
total_entities = sum(len(kg['entities']) for kg in all_article_kgs)
total_relations = sum(len(kg['relations']) for kg in all_article_kgs)

print(f"\nTotal entities extracted: {total_entities}")
print(f"Total relations extracted: {total_relations}")

with_graphs = [kg for kg in all_article_kgs if kg['num_edges'] > 0]
if with_graphs:
    avg_nodes = sum(kg['num_nodes'] for kg in with_graphs) / len(with_graphs)
    avg_edges = sum(kg['num_edges'] for kg in with_graphs) / len(with_graphs)
    print(f"Average nodes per KG: {avg_nodes:.2f}")
    print(f"Average edges per KG: {avg_edges:.2f}")

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import json
import networkx as nx
from typing import List, Dict, Set, Tuple
import dspy
from dspy.signatures import Signature
from dspy import OutputField, InputField
import pandas as pd
import os
from collections import Counter
import numpy as np

print("✓ All imports successful!")

In [ ]:
# ============================================================================
# SAVE KNOWLEDGE GRAPHS TO JSON FILE
# ============================================================================

import json
import networkx as nx

def save_knowledge_graphs(all_article_kgs, filepath='/kaggle/working/article_knowledge_graphs.json'):
    """
    Save knowledge graphs to JSON file.
    
    Args:
        all_article_kgs: List of dictionaries containing KG information
        filepath: Path to save the JSON file
    """
    # Convert NetworkX graphs to serializable format
    serializable_kgs = []
    
    for kg_data in all_article_kgs:
        # Convert NetworkX graph to node-link format
        graph = kg_data['graph']
        graph_dict = nx.node_link_data(graph) if graph.number_of_nodes() > 0 else None
        
        serializable_kg = {
            'id': kg_data['id'],
            'text': kg_data['text'],
            'entities': kg_data['entities'],
            'relations': kg_data['relations'],
            'num_nodes': kg_data['num_nodes'],
            'num_edges': kg_data['num_edges'],
            'graph_data': graph_dict  # Serialized graph
        }
        
        serializable_kgs.append(serializable_kg)
    
    # Save to JSON
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(serializable_kgs, f, indent=2, ensure_ascii=False)
    
    print(f"\n{'='*80}")
    print("KNOWLEDGE GRAPHS SAVED")
    print(f"{'='*80}")
    print(f"Saved {len(serializable_kgs)} knowledge graphs to:")
    print(f"  {filepath}")
    print(f"\nFile size: {len(json.dumps(serializable_kgs)) / (1024*1024):.2f} MB")
    print(f"Articles with graphs: {sum(1 for kg in serializable_kgs if kg['num_edges'] > 0)}")
    print(f"Articles without graphs: {sum(1 for kg in serializable_kgs if kg['num_edges'] == 0)}")

# Save the knowledge graphs
save_knowledge_graphs(all_article_kgs, '/kaggle/working/article_knowledge_graphs.json')

print("\n✓ Knowledge graphs saved successfully!")
print("You can now proceed to the topic classification step.")

In [ ]:
# ============================================================================
# SAVE KNOWLEDGE GRAPHS (PERMANENT STORAGE)
# ============================================================================

import json
import networkx as nx

def save_knowledge_graphs(all_article_kgs, filepath='/kaggle/working/article_knowledge_graphs.json'):
    """Save knowledge graphs to JSON file."""
    serializable_kgs = []
    
    for kg_data in all_article_kgs:
        graph = kg_data['graph']
        graph_dict = nx.node_link_data(graph) if graph.number_of_nodes() > 0 else None
        
        serializable_kg = {
            'id': kg_data['id'],
            'text': kg_data['text'],
            'entities': kg_data['entities'],
            'relations': kg_data['relations'],
            'num_nodes': kg_data['num_nodes'],
            'num_edges': kg_data['num_edges'],
            'graph_data': graph_dict
        }
        serializable_kgs.append(serializable_kg)
    
    # Save to JSON
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(serializable_kgs, f, indent=2, ensure_ascii=False)
    
    print(f"\n{'='*80}")
    print("KNOWLEDGE GRAPHS SAVED")
    print(f"{'='*80}")
    print(f"Saved {len(serializable_kgs)} knowledge graphs to:")
    print(f"  {filepath}")
    print(f"\nFile size: {len(json.dumps(serializable_kgs)) / (1024*1024):.2f} MB")
    print(f"Articles with graphs: {sum(1 for kg in serializable_kgs if kg['num_edges'] > 0)}")

# Save the knowledge graphs
save_knowledge_graphs(all_article_kgs, '/kaggle/working/article_knowledge_graphs.json')

print("\n✓ Knowledge graphs saved successfully!")

In [ ]:
# ============================================================================
# SAVE KNOWLEDGE GRAPHS
# ============================================================================

import json
import pickle
import gzip
from pathlib import Path
from datetime import datetime

def save_knowledge_graphs(all_article_kgs, 
                          base_dir="/kaggle/working/saved_kgs",
                          dataset_name="digital_camera"):
    """
    Save knowledge graphs in multiple formats for future use.
    
    Args:
        all_article_kgs: Your list of KG dictionaries
        base_dir: Directory to save files
        dataset_name: Identifier for this dataset
    """
    
    # Create directory
    save_path = Path(base_dir)
    save_path.mkdir(exist_ok=True, parents=True)
    
    print("="*80)
    print("SAVING KNOWLEDGE GRAPHS")
    print("="*80)
    
    # ========================================================================
    # 1. JSON FORMAT (Human-readable, portable)
    # ========================================================================
    
    print("\n1. Saving as JSON...")
    
    serializable_kgs = []
    for kg_data in all_article_kgs:
        graph = kg_data['graph']
        
        # Convert NetworkX graph to dict format
        if graph.number_of_nodes() > 0:
            graph_dict = nx.node_link_data(graph)
        else:
            graph_dict = None
        
        serializable_kg = {
            'id': kg_data['id'],
            'text': kg_data['text'],
            'entities': kg_data['entities'],
            'relations': kg_data['relations'],
            'num_nodes': kg_data['num_nodes'],
            'num_edges': kg_data['num_edges'],
            'graph_data': graph_dict
        }
        serializable_kgs.append(serializable_kg)
    
    # Save with metadata
    output = {
        'metadata': {
            'dataset': dataset_name,
            'created_at': datetime.now().isoformat(),
            'total_graphs': len(serializable_kgs),
            'graphs_with_edges': sum(1 for kg in serializable_kgs if kg['num_edges'] > 0),
            'total_entities': sum(len(kg['entities']) for kg in serializable_kgs),
            'total_relations': sum(len(kg['relations']) for kg in serializable_kgs)
        },
        'knowledge_graphs': serializable_kgs
    }
    
    json_path = save_path / f"{dataset_name}_kgs.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    json_size = json_path.stat().st_size / (1024 * 1024)
    print(f"   ✓ JSON saved: {json_path}")
    print(f"     Size: {json_size:.2f} MB")
    
    # ========================================================================
    # 2. COMPRESSED PICKLE (Fast loading, smaller size)
    # ========================================================================
    
    print("\n2. Saving as compressed pickle...")
    
    pickle_path = save_path / f"{dataset_name}_kgs.pkl.gz"
    with gzip.open(pickle_path, 'wb') as f:
        pickle.dump(all_article_kgs, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    pickle_size = pickle_path.stat().st_size / (1024 * 1024)
    print(f"   ✓ Pickle saved: {pickle_path}")
    print(f"     Size: {pickle_size:.2f} MB")
    
    # ========================================================================
    # 3. SUMMARY FILE
    # ========================================================================
    
    print("\n3. Saving summary...")
    
    summary = {
        'dataset': dataset_name,
        'saved_at': datetime.now().isoformat(),
        'total_articles': len(all_article_kgs),
        'articles_with_kgs': sum(1 for kg in all_article_kgs if kg['num_edges'] > 0),
        'total_entities': sum(len(kg['entities']) for kg in all_article_kgs),
        'total_relations': sum(len(kg['relations']) for kg in all_article_kgs),
        'files': {
            'json': str(json_path),
            'pickle': str(pickle_path)
        },
        'file_sizes_mb': {
            'json': round(json_size, 2),
            'pickle': round(pickle_size, 2)
        }
    }
    
    summary_path = save_path / f"{dataset_name}_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"   ✓ Summary saved: {summary_path}")
    
    # ========================================================================
    # FINAL REPORT
    # ========================================================================
    
    print("\n" + "="*80)
    print("✓ SAVE COMPLETE!")
    print("="*80)
    print(f"\nDataset: {dataset_name}")
    print(f"Location: {base_dir}")
    print(f"\nFiles created:")
    print(f"  1. {json_path.name} ({json_size:.2f} MB)")
    print(f"  2. {pickle_path.name} ({pickle_size:.2f} MB)")
    print(f"  3. {summary_path.name}")
    print(f"\nStatistics:")
    print(f"  Total articles: {len(all_article_kgs)}")
    print(f"  Articles with KGs: {summary['articles_with_kgs']}")
    print(f"  Total entities: {summary['total_entities']}")
    print(f"  Total relations: {summary['total_relations']}")
    
    return summary


def load_knowledge_graphs(base_dir="/kaggle/working/saved_kgs",
                          dataset_name="digital_camera",
                          format="pickle"):
    """
    Load previously saved knowledge graphs.
    
    Args:
        base_dir: Directory where KGs are saved
        dataset_name: Dataset identifier
        format: 'pickle' (faster) or 'json' (portable)
    
    Returns:
        List of KG dictionaries with reconstructed graphs
    """
    
    save_path = Path(base_dir)
    
    print("="*80)
    print("LOADING KNOWLEDGE GRAPHS")
    print("="*80)
    print(f"\nDataset: {dataset_name}")
    print(f"Format: {format}")
    
    if format == "pickle":
        # ====================================================================
        # LOAD FROM PICKLE (FASTEST)
        # ====================================================================
        
        pickle_path = save_path / f"{dataset_name}_kgs.pkl.gz"
        
        if not pickle_path.exists():
            raise FileNotFoundError(f"Pickle file not found: {pickle_path}")
        
        print(f"\nLoading from: {pickle_path}")
        
        with gzip.open(pickle_path, 'rb') as f:
            all_article_kgs = pickle.load(f)
        
        print(f"✓ Loaded {len(all_article_kgs)} knowledge graphs")
        
    else:
        # ====================================================================
        # LOAD FROM JSON
        # ====================================================================
        
        json_path = save_path / f"{dataset_name}_kgs.json"
        
        if not json_path.exists():
            raise FileNotFoundError(f"JSON file not found: {json_path}")
        
        print(f"\nLoading from: {json_path}")
        
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        metadata = data.get('metadata', {})
        serialized_kgs = data['knowledge_graphs']
        
        print(f"Created: {metadata.get('created_at', 'unknown')}")
        
        # Reconstruct NetworkX graphs
        all_article_kgs = []
        
        for kg_data in serialized_kgs:
            graph_dict = kg_data.get('graph_data')
            
            if graph_dict:
                graph = nx.node_link_graph(graph_dict, directed=True)
            else:
                graph = nx.DiGraph()
            
            reconstructed_kg = {
                'id': kg_data['id'],
                'text': kg_data['text'],
                'entities': kg_data['entities'],
                'relations': kg_data['relations'],
                'num_nodes': kg_data['num_nodes'],
                'num_edges': kg_data['num_edges'],
                'graph': graph
            }
            
            all_article_kgs.append(reconstructed_kg)
        
        print(f"✓ Reconstructed {len(all_article_kgs)} knowledge graphs")
    
    # ========================================================================
    # VERIFICATION
    # ========================================================================
    
    graphs_with_edges = sum(1 for kg in all_article_kgs if kg['num_edges'] > 0)
    total_entities = sum(len(kg['entities']) for kg in all_article_kgs)
    total_relations = sum(len(kg['relations']) for kg in all_article_kgs)
    
    print("\n" + "="*80)
    print("✓ LOAD COMPLETE!")
    print("="*80)
    print(f"\nStatistics:")
    print(f"  Total graphs: {len(all_article_kgs)}")
    print(f"  Graphs with edges: {graphs_with_edges}")
    print(f"  Total entities: {total_entities}")
    print(f"  Total relations: {total_relations}")
    
    return all_article_kgs


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# After creating KGs, save them:
"""
summary = save_knowledge_graphs(
    all_article_kgs,
    base_dir="/kaggle/working/saved_kgs",
    dataset_name="digital_camera_v2"
)
"""

# Later, in a new session, load them:
"""
all_article_kgs = load_knowledge_graphs(
    base_dir="/kaggle/working/saved_kgs",
    dataset_name="digital_camera_v2",
    format="pickle"  # Use "pickle" for speed, "json" for portability
)

# Now use all_article_kgs for any experiments!
# No need to regenerate the graphs!
"""

In [ ]:
# Save the knowledge graphs
summary = save_knowledge_graphs(
    all_article_kgs,
    base_dir="/kaggle/working/saved_kgs",
    dataset_name="digital_camera_2"
)

In [ ]:
# ============================================================================
# EXTRACT ALL UNIQUE CONCEPTS FROM DATASET
# ============================================================================

def extract_unique_concepts(articles: List[Dict]) -> List[str]:
    """
    Extract all unique concepts from the dataset's 'Concept' field.
    
    Args:
        articles: List of article dictionaries
    
    Returns:
        Sorted list of unique concepts
    """
    all_concepts = set()
    
    for article in articles:
        # Get concepts from the Concept field
        if 'Concept' in article and article['Concept']:
            if isinstance(article['Concept'], list):
                all_concepts.update([c.lower().strip() for c in article['Concept'] if c])
            elif isinstance(article['Concept'], str):
                all_concepts.add(article['Concept'].lower().strip())
    
    # Convert to sorted list
    unique_concepts = sorted(list(all_concepts))
    
    print(f"\n{'='*80}")
    print(f"UNIQUE CONCEPTS EXTRACTED FROM DATASET")
    print(f"{'='*80}")
    print(f"Total unique concepts: {len(unique_concepts)}")
    print(f"\nFirst 20 concepts: {unique_concepts[:20]}")
    print(f"\nLast 20 concepts: {unique_concepts[-20:]}")
    
    return unique_concepts


# Option 1: Extract concepts from 'data'
all_concepts = extract_unique_concepts(data)

# Save concept list for reference
with open('/kaggle/working/all_concepts.json', 'w') as f:
    json.dump(all_concepts, f, indent=2)

print(f"\n✓ Saved concept list to /kaggle/working/all_concepts.json")
print(f"✓ Ready for topic classification with {len(all_concepts)} concepts")

In [ ]:
# ============================================================================
# STEP 1: LOAD KEYWORD MAPPINGS
# ============================================================================

def load_keyword_mappings(filepath: str) -> Dict[str, List[str]]:
    """
    Load keyword mappings for each concept/topic.
    
    Expected format (list of objects where first keyword is the concept):
    [
        {"Keyword": ["Addiction", "opioids", "alcohol", "drug", ...]},
        {"Keyword": ["Alcohol", "wine", "consumption", ...]},
        ...
    ]
    
    Returns:
    {
        "addiction": ["opioids", "alcohol", "drug", "overdose", ...],
        "alcohol": ["wine", "consumption", "addiction", ...],
        ...
    }
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        keyword_data = json.load(f)
    
    # Convert to dictionary: concept -> keywords
    keyword_map = {}
    
    for item in keyword_data:
        keywords_list = item.get('Keyword', [])
        
        if len(keywords_list) > 0:
            # First keyword is the concept name
            concept = keywords_list[0].lower().strip()
            # Rest are the associated keywords
            associated_keywords = [kw.lower().strip() for kw in keywords_list[1:]]
            
            keyword_map[concept] = associated_keywords
    
    print(f"✓ Loaded keyword mappings for {len(keyword_map)} concepts")
    print(f"  Sample: {list(keyword_map.keys())[:5]}")
    
    return keyword_map

In [ ]:
# ============================================================================
# FORMAT TOPICS WITH KEYWORDS FOR LLM
# ============================================================================

def format_topics_with_keywords(topics: List[str], keyword_map: Dict[str, List[str]]) -> str:
    """
    Format topics with their associated keywords for LLM context.
    """
    formatted_topics = []
    
    for topic in topics:
        topic_lower = topic.lower()
        keywords = keyword_map.get(topic_lower, [])
        
        if keywords:        
            keywords_str = ", ".join(keywords)
            formatted_topics.append(f"{topic} (keywords: {keywords_str})")
        else:
            formatted_topics.append(topic)
    
    return "\n".join(formatted_topics)

print("✓ Keyword formatting function ready")

In [ ]:
# ============================================================================
# LOAD DATA AND EXISTING KNOWLEDGE GRAPHS
# ============================================================================

def load_json_data(filepath: str) -> List[Dict]:
    """Load JSON data from file"""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✓ Loaded {len(data)} documents from {filepath}")
    return data

def load_knowledge_graphs(filepath: str) -> List[Dict]:
    """Load previously created knowledge graphs"""
    if not os.path.exists(filepath):
        print(f"\n⚠️  WARNING: Knowledge graph file not found at {filepath}")
        return None
    
    with open(filepath, 'r', encoding='utf-8') as f:
        kgs = json.load(f)
    print(f"✓ Loaded {len(kgs)} knowledge graphs from {filepath}")
    return kgs

# Load data
articles = load_json_data('DATASET_PATH')   #put dataset path
knowledge_graphs = load_knowledge_graphs('/kaggle/working/article_knowledge_graphs.json')

# Only classify articles that have KGs
articles_with_kgs = articles[:len(knowledge_graphs)] 

if knowledge_graphs is None:
    print("\n❌ Cannot proceed without knowledge graphs. Please save them first.")
else:
    print(f"\n✓ Successfully loaded {len(articles)} articles and {len(knowledge_graphs)} KGs")
    print("✓ Ready to proceed with topic classification!")

In [ ]:
# ============================================================================
# KG FORMATTING FUNCTION
# ============================================================================

def format_kg_for_llm(kg_data: Dict) -> str:
    """Format knowledge graph into readable text for the LLM."""
    if not kg_data or kg_data.get('num_edges', 0) == 0:
        return "No knowledge graph available."
    
    # Format entities
    entities = kg_data.get('entities', [])
    entities_str = ", ".join(entities[:3000]) if entities else "None"
    if len(entities) > 3000:
        entities_str += f"... ({len(entities) - 3000} more)"
    
    # Format relationships
    relations = kg_data.get('relations', [])
    if relations:
        relationships = []
        for relation in relations[:2000]:
            if isinstance(relation, (list, tuple)) and len(relation) >= 3:
                source, rel_type, target = relation[0], relation[1], relation[2]
                relationships.append(f"{source} --[{rel_type}]--> {target}")
        relationships_str = "\n".join(relationships)
        if len(relations) > 2000:
            relationships_str += f"\n... ({len(relations) - 2000} more relationships)"
    else:
        relationships_str = "None"
    
    kg_summary = f"""Knowledge Graph Summary:
Entities: {entities_str}

Key Relationships:
{relationships_str}"""
    
    return kg_summary

print("✓ KG formatting function ready")

In [ ]:
# ============================================================================
# CREATE VARIABLES FOR ALL ARTICLES WITH GROUND TRUTH
# ============================================================================

# Use ALL articles and ALL KGs
first_50_articles = individual_articles  
first_50_kgs = all_article_kgs 

# Add ground truth from original data
print("Adding ground truth labels...")
for idx, article in enumerate(first_50_articles):
    article_id = article['id']
    if article_id < len(data):
        true_concepts = data[article_id].get('Concept', [])
        article['Concept'] = true_concepts if true_concepts else []
    else:
        article['Concept'] = []

print(f"\n✓ Created first_50_articles: {len(first_50_articles)} articles")
print(f"✓ Created first_50_kgs: {len(first_50_kgs)} knowledge graphs")

# Count how many have labels
with_labels = sum(1 for a in first_50_articles if a.get('Concept', []))
print(f"✓ {with_labels}/{len(first_50_articles)} articles have ground truth labels")

In [ ]:
# ============================================================================
# ENHANCED EVALUATION FUNCTION
# ============================================================================

import numpy as np
from typing import List, Dict
import pandas as pd

def evaluate_classification_complete(results: List[Dict], method_name: str = "Classification") -> Dict:
    """
    Calculate both MACRO and MICRO averaged F1 scores.
    
    MACRO F1: Average of per-article F1 scores (treats each article equally)
    MICRO F1: Global F1 from total TP/FP/FN (weights by prediction volume)
    """
    total = len(results)
    exact_match = 0
    partial_match = 0
    no_match = 0
    no_true_labels = 0
    
    # For MACRO-AVERAGED metrics
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    # For MICRO-AVERAGED metrics
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    correct_predictions = []
    incorrect_predictions = []
    
    for result in results:
        true_set = set([c.lower().strip() for c in result['true_concepts']]) if result['true_concepts'] else set()
        pred_set = set(result['predicted_topics'])
        
        if len(true_set) == 0:
            no_true_labels += 1
            continue
        
        intersection = true_set & pred_set
        
        # Calculate TP, FP, FN for this article
        tp = len(intersection)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        # Accumulate for MICRO metrics
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # Match categorization
        if true_set == pred_set and len(true_set) > 0:
            exact_match += 1
            correct_predictions.append(result)
        elif len(intersection) > 0:
            partial_match += 1
        else:
            no_match += 1
            incorrect_predictions.append(result)
        
        # Calculate per-article metrics for MACRO averaging
        precision = tp / len(pred_set) if len(pred_set) > 0 else 0
        recall = tp / len(true_set) if len(true_set) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
    
    articles_with_true_labels = total - no_true_labels
    
    # MACRO-AVERAGED METRICS (average of per-article scores)
    macro_precision = np.mean(all_precisions) if all_precisions else 0
    macro_recall = np.mean(all_recalls) if all_recalls else 0
    macro_f1 = np.mean(all_f1s) if all_f1s else 0
    
    # MICRO-AVERAGED METRICS (global TP, FP, FN)
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0
    
    metrics = {
        'method_name': method_name,
        'total_articles': total,
        'articles_with_true_labels': articles_with_true_labels,
        'articles_without_true_labels': no_true_labels,
        'exact_matches': exact_match,
        'partial_matches': partial_match,
        'no_matches': no_match,
        'exact_match_rate': exact_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'partial_match_rate': partial_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'total_tp': total_tp,
        'total_fp': total_fp,
        'total_fn': total_fn,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1,
        'correct_examples': correct_predictions[:3],
        'incorrect_examples': incorrect_predictions[:3]
    }
    
    # PRINT RESULTS
    print(f"\n{'='*80}")
    print(f"EVALUATION METRICS - {method_name.upper()}")
    print(f"{'='*80}")
    print(f"\nTotal articles evaluated: {articles_with_true_labels}")
    print(f"Articles without labels (skipped): {no_true_labels}")
    
    print(f"\n🎯 CONFUSION MATRIX (Topic-level):")
    print(f"  True Positives (TP):  {total_tp}")
    print(f"  False Positives (FP): {total_fp}")
    print(f"  False Negatives (FN): {total_fn}")
    
    print(f"\n📊 ACCURACY:")
    print(f"  ✓ Exact matches: {exact_match} ({metrics['exact_match_rate']*100:.1f}%)")
    print(f"  ~ Partial matches: {partial_match} ({metrics['partial_match_rate']*100:.1f}%)")
    print(f"  ✗ No matches: {no_match} ({no_match/articles_with_true_labels*100:.1f}%)")
    
    print(f"\n📈 MACRO-AVERAGED METRICS (per-article average):")
    print(f"  Precision: {macro_precision:.4f}")
    print(f"  Recall:    {macro_recall:.4f}")
    print(f"  F1 Score:  {macro_f1:.4f}")
    
    print(f"\n📊 MICRO-AVERAGED METRICS (global):")
    print(f"  Precision: {micro_precision:.4f}")
    print(f"  Recall:    {micro_recall:.4f}")
    print(f"  F1 Score:  {micro_f1:.4f}")
    
    print(f"\n💡 INTERPRETATION:")
    print(f"  - Macro F1 treats each article equally")
    print(f"  - Micro F1 weights by total predictions")
    if macro_f1 > micro_f1:
        print(f"  - Macro > Micro: Better on articles with fewer topics")
    elif micro_f1 > macro_f1:
        print(f"  - Micro > Macro: Better on articles with many topics")
    else:
        print(f"  - Equal performance across article sizes")
    
    return metrics

print("✓ Enhanced evaluation function loaded!")

In [ ]:
# ============================================================================
# SELF-CONSISTENCY CLASSIFIER
# ============================================================================

from collections import Counter

class SelfConsistencyClassifier:
    """Run classification multiple times and take consensus."""
    
    def __init__(self, available_topics: List[str], num_runs: int = 5, 
                 min_votes: int = 2):
        self.available_topics = available_topics
        self.num_runs = num_runs
        self.min_votes = min_votes
        
        class ConsensusClassification(dspy.Signature):
            """Classify topics for this article. Be thoughtful and precise.
            Return 1-3 topics that are clearly discussed in the article.
            """
            
            article_text: str = dspy.InputField(desc="Article content")
            kg_summary: str = dspy.InputField(desc="Knowledge graph summary")
            available_topics: str = dspy.InputField(desc="Possible topics")
            
            predicted_topics: str = dspy.OutputField(
                desc="1-3 relevant topics or 'none'"
            )
        
        self.classifier = dspy.ChainOfThought(ConsensusClassification)
        self.available_topics_str = ", ".join(available_topics)
        print(f"✓ Self-consistency classifier ({num_runs} runs, {min_votes} min votes)")
    
    def classify_article(self, article_text: str, kg_data: Dict) -> Dict:
        """Run classification multiple times and vote."""
        
        kg_summary = format_kg_for_llm(kg_data)
        all_predictions = []
        
        # Run classification num_runs times with temperature
        for run in range(self.num_runs):
            try:
                lm_temp = dspy.LM(
                model="openai/gpt-4o",
                api_key="API_KEY", #put api key
                api_base="https://openrouter.ai/api/v1",
                max_tokens=10000,
                temperature=0.5
                )
                
                old_lm = dspy.settings.lm
                dspy.settings.configure(lm=lm_temp)
                
                result = self.classifier(
                    article_text=article_text,
                    kg_summary=kg_summary,
                    available_topics=self.available_topics_str
                )
                
                dspy.settings.configure(lm=old_lm)
                
                predicted = result.predicted_topics.strip().lower()
                
                if predicted != 'none':
                    topics = [t.strip().lower() for t in predicted.split(',') 
                             if t.strip()]
                    topics = [t for t in topics 
                             if t in [at.lower() for at in self.available_topics]]
                    all_predictions.extend(topics)
                    
            except Exception as e:
                print(f"Run {run} error: {e}")
                continue
        
        # Vote: keep topics appearing in >= min_votes runs
        vote_counts = Counter(all_predictions)
        consensus_topics = [topic for topic, count in vote_counts.items() 
                          if count >= self.min_votes]
        
        return {
            'predicted_topics': consensus_topics[:3],
            'num_topics': len(consensus_topics[:3]),
            'vote_counts': dict(vote_counts)
        }
    
    def classify_dataset(self, articles: List[Dict], knowledge_graphs: List[Dict],
                        max_articles: int = None) -> List[Dict]:
        """Classify with self-consistency."""
        results = []
        kg_dict = {kg['id']: kg for kg in knowledge_graphs}
        
        if max_articles:
            articles = articles[:max_articles]
        
        print(f"\n{'='*80}")
        print(f"SELF-CONSISTENCY CLASSIFICATION")
        print(f"Processing {len(articles)} articles ({self.num_runs} runs each)")
        print(f"⚠️  This will take approximately {len(articles) * self.num_runs * 2 / 60:.0f} minutes")
        print(f"{'='*80}\n")
        
        for idx, article in enumerate(articles):
            print(f"Article {idx}/{len(articles)}...", end=" ", flush=True)
            
            article_text = article.get('Article Text', '')
            kg_data = kg_dict.get(idx, {'num_edges': 0, 'entities': [], 'relations': []})
            classification = self.classify_article(article_text, kg_data)
            
            result = {
                'article_id': idx,
                'article_text': article_text,
                'true_concepts': article.get('Concept', []),
                'predicted_topics': classification['predicted_topics'],
                'num_predicted': classification['num_topics']
            }
            
            results.append(result)
            print(f"✓ {len(classification['predicted_topics'])} topics")
            
            # Progress update every 50 articles
            if (idx + 1) % 50 == 0:
                print(f"\n{'='*80}")
                print(f"Progress: {idx + 1}/{len(articles)} articles processed")
                print(f"{'='*80}\n")
        
        return results


print("✓ Self-Consistency Classifier defined")

In [ ]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_classification(results: List[Dict]) -> Dict:
    """
    Calculate accuracy metrics by comparing predicted topics with true concepts.
    """
    total = len(results)
    exact_match = 0
    partial_match = 0
    no_match = 0
    no_true_labels = 0
    
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    correct_predictions = []
    incorrect_predictions = []
    
    for result in results:
        true_set = set([c.lower().strip() for c in result['true_concepts']]) if result['true_concepts'] else set()
        pred_set = set(result['predicted_topics'])
        
        if len(true_set) == 0:
            no_true_labels += 1
            continue
        
        intersection = true_set & pred_set
        
        tp = len(intersection)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        if true_set == pred_set and len(true_set) > 0:
            exact_match += 1
            correct_predictions.append(result)
        elif len(intersection) > 0:
            partial_match += 1
        else:
            no_match += 1
            incorrect_predictions.append(result)
        
        if len(pred_set) > 0:
            precision = len(intersection) / len(pred_set)
            all_precisions.append(precision)
        else:
            all_precisions.append(0)
        
        if len(true_set) > 0:
            recall = len(intersection) / len(true_set)
            all_recalls.append(recall)
        else:
            all_recalls.append(0)
        
        if all_precisions[-1] + all_recalls[-1] > 0:
            f1 = 2 * (all_precisions[-1] * all_recalls[-1]) / (all_precisions[-1] + all_recalls[-1])
            all_f1s.append(f1)
        else:
            all_f1s.append(0)
    
    articles_with_true_labels = total - no_true_labels
    
    metrics = {
        'total_articles': total,
        'articles_with_true_labels': articles_with_true_labels,
        'articles_without_true_labels': no_true_labels,
        'exact_matches': exact_match,
        'partial_matches': partial_match,
        'no_matches': no_match,
        'exact_match_rate': exact_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'partial_match_rate': partial_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'avg_precision': np.mean(all_precisions) if all_precisions else 0,
        'avg_recall': np.mean(all_recalls) if all_recalls else 0,
        'avg_f1': np.mean(all_f1s) if all_f1s else 0,
        'total_tp': total_tp,
        'total_fp': total_fp,
        'total_fn': total_fn,
        'correct_examples': correct_predictions[:3],
        'incorrect_examples': incorrect_predictions[:3]
    }
    
    print(f"\n{'='*80}")
    print("EVALUATION METRICS")
    print(f"{'='*80}")
    print(f"\nTotal articles evaluated: {articles_with_true_labels}")
    print(f"Articles without labels (skipped): {no_true_labels}")
    
    print(f"\n🎯 CONFUSION MATRIX (Topic-level):")
    print(f"  True Positives (TP):  {total_tp}")
    print(f"  False Positives (FP): {total_fp}")
    print(f"  False Negatives (FN): {total_fn}")
    
    print(f"\n📊 ACCURACY:")
    print(f"  ✓ Exact matches: {exact_match} ({metrics['exact_match_rate']*100:.1f}%)")
    print(f"  ~ Partial matches: {partial_match} ({metrics['partial_match_rate']*100:.1f}%)")
    print(f"  ✗ No matches: {no_match}")
    
    print(f"\n📈 PERFORMANCE METRICS:")
    print(f"  Precision: {metrics['avg_precision']:.3f}")
    print(f"  Recall: {metrics['avg_recall']:.3f}")
    print(f"  F1 Score: {metrics['avg_f1']:.3f}")
    
    return metrics


def save_results(results: List[Dict], output_path: str):
    """Save classification results to JSON file."""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n✓ Results saved to {output_path}")


def export_to_csv(results: List[Dict], output_path: str):
    """Export results to CSV for easy analysis."""
    rows = []
    for r in results:
        rows.append({
            'article_id': r['article_id'],
            'true_concepts': '|'.join(r['true_concepts']) if r['true_concepts'] else '',
            'predicted_topics': '|'.join(r['predicted_topics']),
            'num_predicted': len(r['predicted_topics'])
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✓ CSV exported to {output_path}")


print("✓ Evaluation functions defined")

In [ ]:
# ============================================================================
# RUN SELF-CONSISTENCY ON FULL DATASET
# ============================================================================

print("\n" + "="*80)
print("RUNNING SELF-CONSISTENCY ON FULL DATASET")
print(f"Total articles: {len(first_50_articles)}")
print(f"Total KGs: {len(first_50_kgs)}")
print("="*80)

# Initialize Self-Consistency classifier
classifier = SelfConsistencyClassifier(
    all_concepts, 
    num_runs=3, 
    min_votes=2
)

# Run classification on ALL articles
results = classifier.classify_dataset(
    articles=first_50_articles,  
    knowledge_graphs=first_50_kgs,  
    max_articles=None  # Process ALL articles
)

# Evaluate
print("\n" + "="*80)
print("EVALUATING FULL DATASET")
print("="*80)
eval_metrics = evaluate_classification_complete(results, "Self-Consistency")

# Save results
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

save_results(results, '/kaggle/working/topic_classification_full_dataset.json')
export_to_csv(results, '/kaggle/working/topic_classification_full_dataset.csv')

print(f"\n{'='*80}")
print("✓ COMPLETE!")
print(f"{'='*80}")

In [ ]:
class TopicClassificationWithKeywords(Signature):
    """Given an article text, its knowledge graph, and a list of possible topics 
    with their associated keywords, determine which topics (can be multiple) are 
    most relevant to this article.
    
    Use the keywords to better understand what each topic represents. Match the 
    article content and knowledge graph entities/relations against these keywords.
    
    Return only topic names that are actually present in the available_topics list. 
    Return NO MORE THAN 4 topics in total. If no topics meet the strong-evidence 
    criteria, return `None`."""
    
    article_text: str = InputField(
        desc="The text content of the article"
    )
    knowledge_graph: str = InputField(
        desc="Knowledge graph extracted from the article showing entities and relationships"
    )
    available_topics_with_keywords: str = InputField(
        desc="List of all possible topics with their associated keywords. Format: 'topic (keywords: keyword1, keyword2, ...)'"
    )
    
    predicted_topics: str = OutputField(
        desc="Comma-separated list of relevant topics from the available_topics list. "
             "Only return topic NAMES (not keywords) from the available_topics list. "
             "If multiple topics apply, list them all (maximum 4). If no topics match well, return 'none'."
    )

print("✓ Enhanced topic classification signature defined")

In [ ]:
class ArticleTopicClassifierWithKeywords:
    """Multi-label topic classifier using article text, KG, topics, and keywords."""
    
    def __init__(self, available_topics: List[str], keyword_mappings: Dict[str, List[str]]):
        self.available_topics = available_topics
        self.keyword_mappings = keyword_mappings
        
        # Format topics with keywords for LLM
        self.topics_with_keywords_str = format_topics_with_keywords(
            available_topics, 
            keyword_mappings
        )
        
        self.classifier = dspy.ChainOfThought(TopicClassificationWithKeywords)
        print(f"✓ Classifier initialized with {len(available_topics)} topics and keywords")
    
    def classify_article(self, article_text: str, kg_data: Dict) -> Dict:
        """Classify a single article using keywords."""
        kg_summary = format_kg_for_llm(kg_data)
        
        try:
            result = self.classifier(
                article_text=article_text,
                knowledge_graph=kg_summary,
                available_topics_with_keywords=self.topics_with_keywords_str
            )
            
            predicted_topics_raw = result.predicted_topics
            
            if predicted_topics_raw.lower() == 'none':
                predicted_topics = []
            else:
                predicted_topics = [
                    t.strip().lower() 
                    for t in predicted_topics_raw.split(',')
                    if t.strip()
                ]
                predicted_topics = [
                    t for t in predicted_topics 
                    if t in [at.lower() for at in self.available_topics]
                ]
            
            return {
                'predicted_topics': predicted_topics,
                'num_topics': len(predicted_topics)
            }
            
        except Exception as e:
            print(f"  Error: {str(e)}")
            return {
                'predicted_topics': [],
                'num_topics': 0
            }
    
    def classify_dataset(self, articles: List[Dict], knowledge_graphs: List[Dict],
                        max_articles: int = None) -> List[Dict]:
        """Classify multiple articles."""
        results = []
        kg_dict = {kg['id']: kg for kg in knowledge_graphs}
        
        if max_articles:
            articles = articles[:max_articles]
        
        print(f"\n{'='*80}")
        print(f"CLASSIFYING {len(articles)} ARTICLES WITH KEYWORD-ENHANCED MODEL")
        print(f"{'='*80}\n")
        
        for idx, article in enumerate(articles):
            print(f"Processing article {idx}...", end=" ")
            
            article_text = article.get('Article Text', '')
            kg_data = kg_dict.get(idx, {'num_edges': 0})
            classification = self.classify_article(article_text, kg_data)
            
            result = {
                'article_id': idx,
                'article_text': article_text,
                'true_concepts': article.get('Concept', []),
                'predicted_topics': classification['predicted_topics'],
                'num_predicted': classification['num_topics']
            }
            
            results.append(result)
            print(f"✓ Found {len(classification['predicted_topics'])} topics")
        
        return results

print("✓ Enhanced classifier class defined")

In [ ]:
# ============================================================================
# COMPLETE KEYWORD LOADING 
# ============================================================================

import json
from typing import Dict, List

def load_keyword_mappings(filepath: str) -> Dict[str, List[str]]:
    """
    Load keyword mappings for each concept/topic.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        keyword_data = json.load(f)
    
    # Convert to dictionary: concept -> keywords
    keyword_map = {}
    
    for item in keyword_data:
        keywords_list = item.get('Keyword', [])
        
        if len(keywords_list) > 0:
            # First keyword is the concept name
            concept = keywords_list[0].lower().strip()
            # Rest are the associated keywords
            associated_keywords = [kw.lower().strip() for kw in keywords_list[1:]]
            
            keyword_map[concept] = associated_keywords
    
    print(f"✓ Loaded keyword mappings for {len(keyword_map)} concepts")
    print(f"  Sample concepts: {list(keyword_map.keys())[:5]}")
    
    return keyword_map

# Load the keywords
keyword_mappings = load_keyword_mappings('ketword_dataset_path') #put keyword dataset path

# Verify it worked
print(f"\n✓ keyword_mappings variable created!")
print(f"  Type: {type(keyword_mappings)}")
print(f"  Number of concepts: {len(keyword_mappings)}")

# Show sample
if len(keyword_mappings) > 0:
    first_concept = list(keyword_mappings.keys())[0]
    print(f"\n  Example: '{first_concept}' -> {keyword_mappings[first_concept][:5]}")

In [ ]:
# Initialize classifier WITH KEYWORDS
classifier = ArticleTopicClassifierWithKeywords(
    available_topics=all_concepts,
    keyword_mappings=keyword_mappings
)

In [ ]:
# ============================================================================
# ANALYSIS FUNCTIONS
# ============================================================================

def analyze_classification_results(results: List[Dict], available_topics: List[str]) -> Dict:
    """Analyze classification results and generate statistics."""
    from collections import Counter
    
    topic_counts = Counter()
    #confidence_dist = Counter()
    
    multi_label_count = 0
    no_label_count = 0
    
    for result in results:
        predicted = result['predicted_topics']
        
        if len(predicted) == 0:
            no_label_count += 1
        elif len(predicted) > 1:
            multi_label_count += 1
        
        for topic in predicted:
            topic_counts[topic] += 1
        
        #confidence_dist[result['confidence']] += 1
    
    stats = {
        'total_articles': len(results),
        'articles_with_topics': len(results) - no_label_count,
        'articles_without_topics': no_label_count,
        'multi_label_articles': multi_label_count,
    }
    
    print(f"\n{'='*80}")
    print("CLASSIFICATION RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"\nTotal articles: {stats['total_articles']}")
    print(f"Articles with topics: {stats['articles_with_topics']} ({stats['articles_with_topics']/stats['total_articles']*100:.1f}%)")
    print(f"Multi-label articles: {stats['multi_label_articles']} ({stats['multi_label_articles']/stats['total_articles']*100:.1f}%)")
    #print(f"Average topics per article: {stats['avg_topics_per_article']:.2f}")
    
    print(f"\n{'='*80}")
    print("TOP 10 MOST FREQUENT TOPICS")
    print(f"{'='*80}")
    
    return stats

def save_results(results: List[Dict], output_path: str):
    """Save classification results to JSON file."""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n✓ Results saved to {output_path}")

def export_to_csv(results: List[Dict], output_path: str):
    """Export results to CSV for easy analysis."""
    rows = []
    for r in results:
        rows.append({
            'article_id': r['article_id'],
            'true_concepts': '|'.join(r['true_concepts']) if r['true_concepts'] else '',
            'predicted_topics': '|'.join(r['predicted_topics']),
            'num_predicted': len(r['predicted_topics']),
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✓ CSV exported to {output_path}")

print("✓ Analysis functions defined")

In [ ]:
# Use all the articles and their KGs
first_50_articles = articles
first_50_kgs = knowledge_graphs

# Initialize classifier
classifier = ArticleTopicClassifierWithKeywords(all_concepts, keyword_mappings)


# Classify all the articles
results = classifier.classify_dataset(
    articles=first_50_articles,
    knowledge_graphs=first_50_kgs,
    max_articles=945000 
)

# Analyze results
stats = analyze_classification_results(results, all_concepts)

# Save results
save_results(results, '/kaggle/working/topic_classification_results_first50.json')
export_to_csv(results, '/kaggle/working/topic_classification_results_first50.csv')

# Show sample results
print(f"\n{'='*80}")
print("SAMPLE RESULTS (First 3)")
print(f"{'='*80}\n")

for result in results[:3]:
    print(f"Article {result['article_id']}:")
    print(f"  Preview: {result['article_text']}")
    print(f"  True: {result['true_concepts']}")
    print(f"  Predicted: {result['predicted_topics']}")
    print()

In [ ]:
def evaluate_classification(results: List[Dict]) -> Dict:
    """
    Calculate accuracy metrics by comparing predicted topics with true concepts.
    """
    total = len(results)
    exact_match = 0
    partial_match = 0
    no_match = 0
    no_true_labels = 0
    
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    # NEW: Track TP, FP, FN across all articles
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    correct_predictions = []
    incorrect_predictions = []
    
    for result in results:
        true_set = set([c.lower().strip() for c in result['true_concepts']]) if result['true_concepts'] else set()
        pred_set = set(result['predicted_topics'])
        
        if len(true_set) == 0:
            no_true_labels += 1
            continue
        
        # Find intersection
        intersection = true_set & pred_set
        
        # NEW: Calculate TP, FP, FN for this article
        tp = len(intersection)  # Correctly predicted
        fp = len(pred_set - true_set)  # Predicted but not true
        fn = len(true_set - pred_set)  # True but not predicted
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # Exact match
        if true_set == pred_set and len(true_set) > 0:
            exact_match += 1
            correct_predictions.append(result)
        elif len(intersection) > 0:
            partial_match += 1
        else:
            no_match += 1
            incorrect_predictions.append(result)
        
        # Calculate precision, recall, F1
        if len(pred_set) > 0:
            precision = len(intersection) / len(pred_set)
            all_precisions.append(precision)
        else:
            all_precisions.append(0)
        
        if len(true_set) > 0:
            recall = len(intersection) / len(true_set)
            all_recalls.append(recall)
        else:
            all_recalls.append(0)
        
        if all_precisions[-1] + all_recalls[-1] > 0:
            f1 = 2 * (all_precisions[-1] * all_recalls[-1]) / (all_precisions[-1] + all_recalls[-1])
            all_f1s.append(f1)
        else:
            all_f1s.append(0)
    
    articles_with_true_labels = total - no_true_labels
    
    metrics = {
        'total_articles': total,
        'articles_with_true_labels': articles_with_true_labels,
        'articles_without_true_labels': no_true_labels,
        'exact_matches': exact_match,
        'partial_matches': partial_match,
        'no_matches': no_match,
        'exact_match_rate': exact_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'partial_match_rate': partial_match / articles_with_true_labels if articles_with_true_labels > 0 else 0,
        'avg_precision': np.mean(all_precisions) if all_precisions else 0,
        'avg_recall': np.mean(all_recalls) if all_recalls else 0,
        'avg_f1': np.mean(all_f1s) if all_f1s else 0,
        'total_tp': total_tp,  # NEW
        'total_fp': total_fp,  # NEW
        'total_fn': total_fn,  # NEW
        'correct_examples': correct_predictions[:3],
        'incorrect_examples': incorrect_predictions[:3]
    }
    
    # Print results
    print(f"\n{'='*80}")
    print("EVALUATION METRICS")
    print(f"{'='*80}")
    print(f"\nTotal articles evaluated: {articles_with_true_labels}")
    print(f"Articles without labels (skipped): {no_true_labels}")
    
    # NEW: Print TP, FP, FN
    print(f"\n🎯 CONFUSION MATRIX (Topic-level):")
    print(f"  True Positives (TP):  {total_tp} (correct predictions)")
    print(f"  False Positives (FP): {total_fp} (predicted but wrong)")
    print(f"  False Negatives (FN): {total_fn} (missed true topics)")
    
    print(f"\n📊 ACCURACY:")
    print(f"  ✓ Exact matches: {exact_match} ({metrics['exact_match_rate']*100:.1f}%)")
    print(f"  ~ Partial matches: {partial_match} ({metrics['partial_match_rate']*100:.1f}%)")
    print(f"  ✗ No matches: {no_match} ({no_match/articles_with_true_labels*100:.1f}%)")
    print(f"\n📈 PERFORMANCE METRICS:")
    print(f"  Precision: {metrics['avg_precision']:.3f} (how many predicted topics were correct)")
    print(f"  Recall: {metrics['avg_recall']:.3f} (how many true topics were found)")
    print(f"  F1 Score: {metrics['avg_f1']:.3f} (overall accuracy)")
    
    return metrics

In [ ]:
# Run evaluation
print("\nEvaluating classification accuracy...")
eval_metrics = evaluate_classification_complete(results, "Keywords-Only")

# Show correct examples
print(f"\n{'='*80}")
print("✓ CORRECTLY CLASSIFIED EXAMPLES")
print(f"{'='*80}")
for ex in eval_metrics['correct_examples']:
    print(f"\nArticle {ex['article_id']}:")
    print(f"  True: {ex['true_concepts']}")
    print(f"  Predicted: {ex['predicted_topics']}")
    print(f"  ✓ EXACT MATCH!")

# Show incorrect examples
print(f"\n{'='*80}")
print("✗ INCORRECTLY CLASSIFIED EXAMPLES")
print(f"{'='*80}")
for ex in eval_metrics['incorrect_examples']:
    print(f"\nArticle {ex['article_id']}:")
    print(f"  True: {ex['true_concepts']}")
    print(f"  Predicted: {ex['predicted_topics']}")

In [ ]:
# ============================================================================
# DEFINE SIGNATURE FOR KG-ONLY CLASSIFICATION
# ============================================================================

class KGOnlyTopicClassification(dspy.Signature):
    """Classify topics for this article using ONLY the article text and its 
    knowledge graph structure. DO NOT use any external keyword information.
    
    Analyze:
    1. The entities mentioned in the knowledge graph
    2. The relationships between entities
    3. The overall semantic structure revealed by the KG
    4. The article text content
    
    Return 1-3 topics from the available topics list that best match the 
    article's content as revealed by its knowledge graph. If no topics 
    strongly match, return 'none'.
    """
    
    article_text = dspy.InputField(desc="The article content")
    kg_summary = dspy.InputField(desc="Knowledge graph showing entities and relationships")
    available_topics = dspy.InputField(desc="List of possible topic labels")
    
    predicted_topics = dspy.OutputField(
        desc="1-3 relevant topics or 'none'. Return ONLY topic names from the available_topics list."
    )

print("✓ KG-Only Signature defined")

In [ ]:
# ============================================================================
# CREATE KG-ONLY CLASSIFIER CLASS
# ============================================================================

class KGOnlyClassifier:
    """Topic classifier using ONLY knowledge graphs and article text."""
    
    def __init__(self, available_topics: List[str]):
        self.available_topics = available_topics
        self.available_topics_str = ", ".join(available_topics)
        self.classifier = dspy.ChainOfThought(KGOnlyTopicClassification)
        
        print(f"✓ KG-Only Classifier initialized with {len(available_topics)} topics")
        print("  📊 Using: Article Text + Knowledge Graph ONLY")
    
    def format_kg(self, kg_data: Dict) -> str:
        """Format KG for LLM input."""
        if not kg_data or kg_data.get('num_edges', 0) == 0:
            return "No knowledge graph available."
        
        entities = kg_data.get('entities', [])
        entities_str = ", ".join(entities[:3000]) if entities else "None"
        if len(entities) > 3000:
            entities_str += f"... ({len(entities) - 3000} more)"
        
        relations = kg_data.get('relations', [])
        if relations:
            relationships = []
            for rel in relations[:2000]:
                if isinstance(rel, (list, tuple)) and len(rel) >= 3:
                    src, rel_type, tgt = rel[0], rel[1], rel[2]
                    relationships.append(f"{src} --[{rel_type}]--> {tgt}")
            relationships_str = "\n".join(relationships)
            if len(relations) > 2000:
                relationships_str += f"\n... ({len(relations) - 2000} more)"
        else:
            relationships_str = "None"
        
        return f"""Knowledge Graph:
Entities: {entities_str}

Relationships:
{relationships_str}"""
    
    def classify_article(self, article_text: str, kg_data: Dict) -> Dict:
        """Classify a single article using KG only."""
        kg_summary = self.format_kg(kg_data)
        
        try:
            result = self.classifier(
                article_text=article_text,
                kg_summary=kg_summary,
                available_topics=self.available_topics_str
            )
            
            predicted_raw = result.predicted_topics.strip().lower()
            
            if predicted_raw == 'none':
                predicted_topics = []
            else:
                predicted_topics = [
                    t.strip().lower() 
                    for t in predicted_raw.split(',')
                    if t.strip()
                ]
                # Filter to valid topics only
                predicted_topics = [
                    t for t in predicted_topics 
                    if t in [at.lower() for at in self.available_topics]
                ]
            
            return {
                'predicted_topics': predicted_topics,
                'num_topics': len(predicted_topics)
            }
            
        except Exception as e:
            print(f"  Error: {str(e)}")
            return {
                'predicted_topics': [],
                'num_topics': 0
            }
    
    def classify_dataset(self, articles: List[Dict], knowledge_graphs: List[Dict],
                        max_articles: int = None) -> List[Dict]:
        """Classify multiple articles using KG only."""
        results = []
        kg_dict = {kg['id']: kg for kg in knowledge_graphs}
        
        if max_articles:
            articles = articles[:max_articles]
        
        print(f"\n{'='*80}")
        print(f"KG-ONLY CLASSIFICATION (NO KEYWORDS)")
        print(f"Processing {len(articles)} articles")
        print(f"{'='*80}\n")
        
        for idx, article in enumerate(articles):
            print(f"Article {idx}...", end=" ")
            
            article_text = article.get('Article Text', '')
            kg_data = kg_dict.get(idx, {'num_edges': 0, 'entities': [], 'relations': []})
            classification = self.classify_article(article_text, kg_data)
            
            result = {
                'article_id': idx,
                'article_text': article_text,
                'true_concepts': article.get('Concept', []),
                'predicted_topics': classification['predicted_topics'],
                'num_predicted': classification['num_topics']
            }
            
            results.append(result)
            print(f"✓ {len(classification['predicted_topics'])} topics")
        
        return results


print("✓ KG-Only Classifier class defined")

In [ ]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_kg_only(results: List[Dict]) -> Dict:
    """Evaluate KG-only classification results."""
    total = len(results)
    exact_match = 0
    partial_match = 0
    no_match = 0
    no_true_labels = 0
    
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    for result in results:
        true_set = set([c.lower().strip() for c in result['true_concepts']]) if result['true_concepts'] else set()
        pred_set = set(result['predicted_topics'])
        
        if len(true_set) == 0:
            no_true_labels += 1
            continue
        
        intersection = true_set & pred_set
        
        tp = len(intersection)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        if true_set == pred_set and len(true_set) > 0:
            exact_match += 1
        elif len(intersection) > 0:
            partial_match += 1
        else:
            no_match += 1
        
        precision = len(intersection) / len(pred_set) if len(pred_set) > 0 else 0
        recall = len(intersection) / len(true_set) if len(true_set) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
    
    articles_with_labels = total - no_true_labels
    
    metrics = {
        'total_articles': total,
        'articles_with_labels': articles_with_labels,
        'exact_matches': exact_match,
        'partial_matches': partial_match,
        'no_matches': no_match,
        'exact_match_rate': exact_match / articles_with_labels if articles_with_labels > 0 else 0,
        'avg_precision': np.mean(all_precisions) if all_precisions else 0,
        'avg_recall': np.mean(all_recalls) if all_recalls else 0,
        'avg_f1': np.mean(all_f1s) if all_f1s else 0,
        'total_tp': total_tp,
        'total_fp': total_fp,
        'total_fn': total_fn
    }

    print(f"\n{'='*80}")
    print("KG-ONLY EVALUATION RESULTS")
    print(f"{'='*80}")
    print(f"\nArticles evaluated: {articles_with_labels}")
    print(f"\n🎯 CONFUSION MATRIX:")
    print(f"  TP: {total_tp} | FP: {total_fp} | FN: {total_fn}")
    print(f"\n📊 ACCURACY:")
    print(f"  ✓ Exact: {exact_match} ({metrics['exact_match_rate']*100:.1f}%)")
    print(f"  ~ Partial: {partial_match} ({partial_match/articles_with_labels*100:.1f}%)")
    print(f"  ✗ None: {no_match} ({no_match/articles_with_labels*100:.1f}%)")
    print(f"\n📈 METRICS:")
    print(f"  Precision: {metrics['avg_precision']:.3f}")
    print(f"  Recall: {metrics['avg_recall']:.3f}")
    print(f"  F1 Score: {metrics['avg_f1']:.3f}")
    
    return metrics


print("✓ Evaluation function defined")

In [ ]:
# ============================================================================
# RUN KG-ONLY CLASSIFICATION
# ============================================================================

print("\n" + "="*80)
print("RUNNING KG-ONLY CLASSIFICATION")
print("="*80)

# 1. Initialize classifier (NO keywords needed!)
kg_classifier = KGOnlyClassifier(all_concepts)

# 2. Classify all articles
kg_only_results = kg_classifier.classify_dataset(
    articles=first_50_articles,
    knowledge_graphs=first_50_kgs
)

# 3. Evaluate
print("\n" + "="*80)
print("EVALUATING KG-ONLY RESULTS")
print("="*80)
kg_metrics = evaluate_classification_complete(kg_only_results, "KG-Only")

# 4. Save results
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

with open('/kaggle/working/kg_only_results.json', 'w') as f:
    json.dump(kg_only_results, f, indent=2)
print("✓ Saved to /kaggle/working/kg_only_results.json")

# Export to CSV
kg_df = pd.DataFrame([{
    'article_id': r['article_id'],
    'true_concepts': '|'.join(r['true_concepts']) if r['true_concepts'] else '',
    'predicted_topics': '|'.join(r['predicted_topics']),
    'num_predicted': r['num_predicted']
} for r in kg_only_results])
kg_df.to_csv('/kaggle/working/kg_only_results.csv', index=False)
print("✓ Saved to /kaggle/working/kg_only_results.csv")

# 5. Show sample results
print("\n" + "="*80)
print("SAMPLE RESULTS (First 5)")
print("="*80)
for i, result in enumerate(kg_only_results[:5]):
    print(f"\nArticle {result['article_id']}:")
    print(f"  Text: {result['article_text']}...")
    print(f"  True: {result['true_concepts']}")
    print(f"  Predicted: {result['predicted_topics']}")
    match = "✓ EXACT" if set(result['true_concepts']) == set(result['predicted_topics']) else "~ PARTIAL" if set(result['true_concepts']) & set(result['predicted_topics']) else "✗ MISS"
    print(f"  {match}")

print("\n" + "="*80)
print("✓ KG-ONLY CLASSIFICATION COMPLETE!")

In [ ]:
# ============================================================================
# SELF-CONSISTENCY CLASSIFIER WITH KEYWORDS
# Combines: Multiple runs + Voting + Keyword guidance
# ============================================================================

import json
import dspy
import numpy as np
from typing import List, Dict
from collections import Counter

# ============================================================================
# SIGNATURE WITH KEYWORDS
# ============================================================================

class ConsensusClassificationWithKeywords(dspy.Signature):
    """Classify topics for this article using the provided keywords as guidance.
    
    The keywords help you understand what each topic represents. Match the 
    article content and knowledge graph against these keywords.
    
    Be thoughtful and precise. Return 1-3 topics that are CLEARLY discussed 
    in the article. If uncertain, be conservative.
    """
    
    article_text: str = dspy.InputField(desc="Article content")
    kg_summary: str = dspy.InputField(desc="Knowledge graph summary")
    available_topics_with_keywords: str = dspy.InputField(
        desc="Possible topics with their keywords. Format: 'topic (keywords: kw1, kw2, ...)'"
    )
    
    predicted_topics: str = dspy.OutputField(
        desc="1-3 relevant topics or 'none'. Return ONLY topic names from available_topics."
    )


# ============================================================================
# SELF-CONSISTENCY WITH KEYWORDS CLASSIFIER
# ============================================================================

class SelfConsistencyWithKeywords:
    """
    Run classification multiple times with keyword guidance and take consensus.
    
    Best of both worlds:
    - Keywords help LLM understand what each topic means
    - Multiple runs with voting improve reliability
    """
    
    def __init__(self, 
                 available_topics: List[str],
                 keyword_mappings: Dict[str, List[str]],
                 num_runs: int = 5, 
                 min_votes: int = 2):
        """
        Initialize the classifier.
        
        Args:
            available_topics: List of possible topic labels
            keyword_mappings: Dict mapping topics to their keywords
            num_runs: How many times to run classification per article
            min_votes: Minimum votes needed to include a topic
        """
        self.available_topics = available_topics
        self.keyword_mappings = keyword_mappings
        self.num_runs = num_runs
        self.min_votes = min_votes
        
        # Format topics with keywords
        self.topics_with_keywords_str = self._format_topics_with_keywords()
        
        # Create classifier
        self.classifier = dspy.ChainOfThought(ConsensusClassificationWithKeywords)
        
        print(f"✓ Self-Consistency with Keywords Classifier initialized")
        print(f"  Topics: {len(available_topics)}")
        print(f"  Runs per article: {num_runs}")
        print(f"  Min votes required: {min_votes}")
    
    def _format_topics_with_keywords(self) -> str:
        """Format topics with their associated keywords for LLM context."""
        formatted_topics = []
        
        for topic in self.available_topics:
            topic_lower = topic.lower()
            keywords = self.keyword_mappings.get(topic_lower, [])
            
            if keywords:
                # Limit to first 10 keywords to avoid token overflow
                keywords_str = ", ".join(keywords[:10])
                formatted_topics.append(f"{topic} (keywords: {keywords_str})")
            else:
                formatted_topics.append(topic)
        
        return "\n".join(formatted_topics)
    
    def _format_kg(self, kg_data: Dict) -> str:
        """Format knowledge graph for LLM input."""
        if not kg_data or kg_data.get('num_edges', 0) == 0:
            return "No knowledge graph available."
        
        # Format entities
        entities = kg_data.get('entities', [])
        entities_str = ", ".join(entities[:3000]) if entities else "None"
        if len(entities) > 3000:
            entities_str += f"... ({len(entities) - 3000} more)"
        
        # Format relationships
        relations = kg_data.get('relations', [])
        if relations:
            relationships = []
            for rel in relations[:2000]:
                if isinstance(rel, (list, tuple)) and len(rel) >= 3:
                    src, rel_type, tgt = rel[0], rel[1], rel[2]
                    relationships.append(f"{src} --[{rel_type}]--> {tgt}")
            relationships_str = "\n".join(relationships)
            if len(relations) > 2000:
                relationships_str += f"\n... ({len(relations) - 2000} more)"
        else:
            relationships_str = "None"
        
        return f"""Knowledge Graph:
Entities: {entities_str}

Relationships:
{relationships_str}"""
    
    def classify_article(self, article_text: str, kg_data: Dict) -> Dict:
        """
        Run classification multiple times with keywords and vote.
        
        Args:
            article_text: The article content
            kg_data: Knowledge graph data
            
        Returns:
            Dict with predicted topics and vote counts
        """
        kg_summary = self._format_kg(kg_data)
        all_predictions = []
        
        # Run classification num_runs times
        for run in range(self.num_runs):
            try:
                # Use temperature for diversity across runs
                lm_temp = dspy.LM(
                model="openai/gpt-4o",
                api_key="API_KEY",  #put api key
                api_base="https://openrouter.ai/api/v1",
                max_tokens=10000,
                temperature=0.5
                )
                # Temporarily switch to this LM
                old_lm = dspy.settings.lm
                dspy.settings.configure(lm=lm_temp)
                
                # Run classification
                result = self.classifier(
                    article_text=article_text,
                    kg_summary=kg_summary,
                    available_topics_with_keywords=self.topics_with_keywords_str
                )
                
                # Restore original LM
                dspy.settings.configure(lm=old_lm)
                
                # Parse predictions
                predicted = result.predicted_topics.strip().lower()
                
                if predicted != 'none':
                    # Split by comma and clean
                    topics = [t.strip().lower() for t in predicted.split(',') if t.strip()]
                    
                    # Filter to valid topics only
                    valid_topics = [
                        t for t in topics 
                        if t in [at.lower() for at in self.available_topics]
                    ]
                    
                    all_predictions.extend(valid_topics)
                
            except Exception as e:
                print(f"  Warning: Run {run+1} failed - {str(e)[:5000]}")
                continue
        
        # Vote: keep topics appearing in >= min_votes runs
        vote_counts = Counter(all_predictions)
        consensus_topics = [
            topic for topic, count in vote_counts.items() 
            if count >= self.min_votes
        ]
        
        # Limit to top 3 by vote count
        consensus_topics = sorted(
            consensus_topics, 
            key=lambda t: vote_counts[t], 
            reverse=True
        )[:3]
        
        return {
            'predicted_topics': consensus_topics,
            'num_topics': len(consensus_topics),
            'vote_counts': dict(vote_counts),
            'total_predictions': len(all_predictions)
        }
    
    def classify_dataset(self, 
                        articles: List[Dict], 
                        knowledge_graphs: List[Dict],
                        max_articles: int = None) -> List[Dict]:
        """
        Classify multiple articles with self-consistency and keywords.
        
        Args:
            articles: List of article dictionaries
            knowledge_graphs: List of KG dictionaries
            max_articles: Optional limit on number of articles
            
        Returns:
            List of classification results
        """
        results = []
        kg_dict = {kg['id']: kg for kg in knowledge_graphs}
        
        if max_articles:
            articles = articles[:max_articles]
        
        print(f"\n{'='*80}")
        print(f"SELF-CONSISTENCY WITH KEYWORDS CLASSIFICATION")
        print(f"{'='*80}")
        print(f"Processing {len(articles)} articles")
        print(f"Runs per article: {self.num_runs}")
        print(f"⚠️  Estimated time: ~{len(articles) * self.num_runs * 2 / 60:.0f} minutes")
        print(f"{'='*80}\n")
        
        for idx, article in enumerate(articles):
            print(f"Article {idx+1}/{len(articles)}...", end=" ", flush=True)
            
            article_text = article.get('Article Text', '')
            kg_data = kg_dict.get(idx, {
                'num_edges': 0, 
                'entities': [], 
                'relations': []
            })
            
            # Classify with self-consistency + keywords
            classification = self.classify_article(article_text, kg_data)
            
            # Store result
            result = {
                'article_id': idx,
                'article_text': article_text,
                'true_concepts': article.get('Concept', []),
                'predicted_topics': classification['predicted_topics'],
                'num_predicted': classification['num_topics'],
                'vote_counts': classification['vote_counts']
            }
            
            results.append(result)
            print(f"✓ {len(classification['predicted_topics'])} topics (from {classification['total_predictions']} predictions)")
            
            # Progress update every 50 articles
            if (idx + 1) % 50 == 0:
                print(f"\n{'='*80}")
                print(f"Progress: {idx + 1}/{len(articles)} articles")
                print(f"{'='*80}\n")
        
        print(f"\n{'='*80}")
        print("✓ CLASSIFICATION COMPLETE!")
        print(f"{'='*80}")
        
        return results


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_self_consistency_keywords(results: List[Dict]) -> Dict:
    """
    Evaluate classification results.
    
    Args:
        results: List of classification result dictionaries
        
    Returns:
        Dictionary of evaluation metrics
    """
    total = len(results)
    exact_match = 0
    partial_match = 0
    no_match = 0
    no_true_labels = 0
    
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    correct_predictions = []
    incorrect_predictions = []
    
    for result in results:
        true_set = set([c.lower().strip() for c in result['true_concepts']]) if result['true_concepts'] else set()
        pred_set = set(result['predicted_topics'])
        
        if len(true_set) == 0:
            no_true_labels += 1
            continue
        
        intersection = true_set & pred_set
        
        # Calculate TP, FP, FN
        tp = len(intersection)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # Categorize match type
        if true_set == pred_set and len(true_set) > 0:
            exact_match += 1
            correct_predictions.append(result)
        elif len(intersection) > 0:
            partial_match += 1
        else:
            no_match += 1
            incorrect_predictions.append(result)
        
        # Calculate metrics
        precision = tp / len(pred_set) if len(pred_set) > 0 else 0
        recall = tp / len(true_set) if len(true_set) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
    
    articles_with_labels = total - no_true_labels
    
    metrics = {
        'total_articles': total,
        'articles_with_labels': articles_with_labels,
        'articles_without_labels': no_true_labels,
        'exact_matches': exact_match,
        'partial_matches': partial_match,
        'no_matches': no_match,
        'exact_match_rate': exact_match / articles_with_labels if articles_with_labels > 0 else 0,
        'partial_match_rate': partial_match / articles_with_labels if articles_with_labels > 0 else 0,
        'avg_precision': np.mean(all_precisions) if all_precisions else 0,
        'avg_recall': np.mean(all_recalls) if all_recalls else 0,
        'avg_f1': np.mean(all_f1s) if all_f1s else 0,
        'total_tp': total_tp,
        'total_fp': total_fp,
        'total_fn': total_fn,
        'correct_examples': correct_predictions[:3],
        'incorrect_examples': incorrect_predictions[:3]
    }
    
    # Print results
    print(f"\n{'='*80}")
    print("EVALUATION METRICS - SELF-CONSISTENCY WITH KEYWORDS")
    print(f"{'='*80}")
    print(f"\nTotal articles evaluated: {articles_with_labels}")
    print(f"Articles without labels (skipped): {no_true_labels}")
    
    print(f"\n🎯 CONFUSION MATRIX (Topic-level):")
    print(f"  True Positives (TP):  {total_tp}")
    print(f"  False Positives (FP): {total_fp}")
    print(f"  False Negatives (FN): {total_fn}")
    
    print(f"\n📊 ACCURACY:")
    print(f"  ✓ Exact matches: {exact_match} ({metrics['exact_match_rate']*100:.1f}%)")
    print(f"  ~ Partial matches: {partial_match} ({metrics['partial_match_rate']*100:.1f}%)")
    print(f"  ✗ No matches: {no_match}")
    
    print(f"\n📈 PERFORMANCE METRICS:")
    print(f"  Precision: {metrics['avg_precision']:.3f}")
    print(f"  Recall: {metrics['avg_recall']:.3f}")
    print(f"  F1 Score: {metrics['avg_f1']:.3f}")
    
    return metrics


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Initialize classifier with keywords
classifier = SelfConsistencyWithKeywords(
    available_topics=all_concepts,
    keyword_mappings=keyword_mappings,
    num_runs=5,        # Run 5 times per article
    min_votes=2        # Need 3/5 votes to include topic
)

# Run classification
results = classifier.classify_dataset(
    articles=first_50_articles,
    knowledge_graphs=first_50_kgs,
    max_articles=None  # Process all
)

# Evaluate
metrics = evaluate_classification_complete(results, "Self-Consistency + Keywords")

# Save results
import json
with open('/kaggle/working/self_consistency_keywords_results.json', 'w') as f:
    json.dump(results, f, indent=2)